# Task 3: Classificatore SVM Lineare basato su Word Embeddings

Questo notebook esplora l'utilizzo di rappresentazioni vettoriali dense della semantica lessicale (**Word Embeddings statici**) estratte dal corpus *itWaC*. Poiché gli embedding sono associati a singole parole, sperimenteremo **4 diverse strategie di aggregazione (pooling)** a livello di documento per mappare i testi in vettori di dimensione fissa ($128$). I vettori risultanti verranno poi classificati tramite una **Support Vector Machine (LinearSVC)**.

In [14]:
import os
import sqlite3
import numpy as np
import pandas as pd

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

## 1. Caricamento dei Dati ed Estrazione del Vocabolario di Embedding

In questa fase carichiamo i dataset preprocessati e configuriamo la connessione al database SQLite contenente gli embedding pre-addestrati di *itWaC* (dimensione $128$). Se non è già presente, esporteremo il database in un file di testo piatto per velocizzare le letture successive.

In [15]:
# Caricamento dei dataframe pre-elaborati
df_train = pd.read_pickle("../data/processed/df_train_processed.pkl")
df_val = pd.read_pickle("../data/processed/df_val_processed.pkl")
df_test = pd.read_pickle("../data/processed/df_test_processed.pkl")

sql_path = '../data/word_embeddings/itwac128.sqlite'
txt_path = '../data/word_embeddings/itwac128.txt'

# Connessione al DB per l'estrazione
con = sqlite3.connect(sql_path)
cur = con.cursor()

In [16]:
# Dump efficiente della tabella in un file TXT strutturato per letture veloci
if not os.path.exists(txt_path):
    print(f"Esportazione degli embedding da SQLite a {txt_path}...")
    with open(txt_path, 'w', encoding='utf-8') as out_file:
        for embedding in cur.execute("SELECT * FROM store"):
            # Esclude l'ultimo elemento se non necessario, unisce con tabulazioni
            str_embedding = [str(el) for el in embedding[:-1]]
            out_file.write('\t'.join(str_embedding) + '\n')
else:
    print(f"File vettori già esistente: {txt_path}, skip esportazione.")
    
# Chiudiamo la connessione al database non più necessaria
con.close()

File vettori già esistente: ../data/word_embeddings/itwac128.txt, skip esportazione.


In [17]:
def load_word_embeddings(path):
    """Carica il file di testo degli embedding in un dizionario Python {parola: array_numpy}."""
    embeddings = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            word = parts[0]
            # Conversione immediata delle componenti in float32 per ottimizzare la RAM
            vector = np.array([float(val) for val in parts[1:]], dtype=np.float32)
            embeddings[word] = vector
    return embeddings

# Caricamento effettivo in memoria
embeddings_dict = load_word_embeddings(txt_path)
print(f"Numero totale di parole nel vocabolario: {len(embeddings_dict):,}")
print(f"Dimensione dichiarata di ciascun embedding: {next(iter(embeddings_dict.values())).shape[0]}")

Numero totale di parole nel vocabolario: 1,247,492
Dimensione dichiarata di ciascun embedding: 128


## 2. Definizione delle Strategie di Vectorization del Documento

### Strategia 1 e 2: Unweighted Average Pooling (Forme Superficiali vs Lemmi)
La rappresentazione del documento viene calcolata come la **media matematica non pesata** di tutti i vettori delle parole presenti nel testo e registrate nel dizionario di embedding. Testeremo questa tecnica sia sulla colonna dei `tokens` (forme grafiche originali) sia sulla colonna dei `lemmas` (forme canonizzate).

In [18]:
def docs_to_mean_embeddings(docs, embeddings, emb_dim=128):
    """Calcola la media non pesata dei vettori delle parole per ciascun documento."""
    doc_vectors = []
    for text_list in docs:
        valid_vectors = [embeddings[token] for token in text_list if token in embeddings]
        
        if len(valid_vectors) > 0:
            mean_vector = np.mean(valid_vectors, axis=0)
        else:
            mean_vector = np.zeros(emb_dim)
        doc_vectors.append(mean_vector)
        
    return doc_vectors

# Estrazione dei vettori basati sulle FORME SUPERFICIALI (Tokens)
df_train['tokens_vector'] = docs_to_mean_embeddings(df_train['tokens'], embeddings_dict)
df_val['tokens_vector'] = docs_to_mean_embeddings(df_val['tokens'], embeddings_dict)
df_test['tokens_vector'] = docs_to_mean_embeddings(df_test['tokens'], embeddings_dict)

# Estrazione dei vettori basati sulle FORME CANONICHE (Lemmi)
df_train['lemmas_vector'] = docs_to_mean_embeddings(df_train['lemmas'], embeddings_dict)
df_val['lemmas_vector'] = docs_to_mean_embeddings(df_val['lemmas'], embeddings_dict)
df_test['lemmas_vector'] = docs_to_mean_embeddings(df_test['lemmas'], embeddings_dict)

### 2.1 Valutazione sul Validation Set: Tokens vs Lemmi (Unweighted)
Eseguiamo un primo screening delle performance sul Validation Set utilizzando la configurazione standard di SVM Lineare.

In [19]:
# Preparazione matrici per la strategia basata sulle Forme
X_train_tokens = np.vstack(df_train['tokens_vector'].values)
X_val_tokens = np.vstack(df_val['tokens_vector'].values)
y_train = df_train['label'].values
y_val = df_val['label'].values

# Addestramento e Validazione
model_tokens = LinearSVC(random_state=42, max_iter=10000)
model_tokens.fit(X_train_tokens, y_train)
pred_val_tokens = model_tokens.predict(X_val_tokens)

print("--- REPORT PERFORMANCE: EMBEDDING DI TOKENS (VALIDATION) ---")
print(classification_report(y_val, pred_val_tokens, zero_division=0))

--- REPORT PERFORMANCE: EMBEDDING DI TOKENS (VALIDATION) ---
              precision    recall  f1-score   support

           0       0.89      0.92      0.91       500
           1       0.92      0.89      0.91       500

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000



In [20]:
# Preparazione matrici per la strategia basata sui Lemmi
X_train_lemmas = np.vstack(df_train['lemmas_vector'].values)
X_val_lemmas = np.vstack(df_val['lemmas_vector'].values)

# Addestramento e Validazione
model_lemmas = LinearSVC(random_state=42, max_iter=10000)
model_lemmas.fit(X_train_lemmas, y_train)
pred_val_lemmas = model_lemmas.predict(X_val_lemmas)

print("--- REPORT PERFORMANCE: EMBEDDING DI LEMMI (VALIDATION) ---")
print(classification_report(y_val, pred_val_lemmas, zero_division=0))

--- REPORT PERFORMANCE: EMBEDDING DI LEMMI (VALIDATION) ---
              precision    recall  f1-score   support

           0       0.91      0.92      0.91       500
           1       0.92      0.91      0.91       500

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000



### Strategia 3: Content-Word Pooling (Filtro Morfosintattico)
Per limitare il rumore introdotto dalle parole funzionali (congiunzioni, articoli, preposizioni), calcoliamo la media limitando l'estrazione alle sole **parole piene** o di contenuto, identificate dai tag morfosintattici dell'Universal Dependencies: **Sostantivi comuns (`NOUN`)**, **Nomi propri (`PROPN`)** e **Verbi (`VERB`)**.

In [21]:
def docs_to_embeddings_pos(docs_tokens, docs_pos, embeddings, emb_dim=128):
    """Calcola la media dei vettori includendo solo NOUN, PROPN e VERB."""
    doc_vectors = []
    
    for text_tokens, text_pos in zip(docs_tokens, docs_pos):
        vectors = []
        
        # Gestione robusta del tipo di dato in ingresso (lista o stringa formattata)
        lista_token = text_tokens if isinstance(text_tokens, list) else str(text_tokens).split()
        lista_pos = text_pos if isinstance(text_pos, list) else str(text_pos).split()

        for token, pos_tag in zip(lista_token, lista_pos):
            if pos_tag in ['NOUN', 'PROPN', 'VERB']:
                if token in embeddings:
                    vectors.append(embeddings[token])

        if len(vectors) > 0:
            mean_vector = np.mean(vectors, axis=0)
        else:
            mean_vector = np.zeros(emb_dim)
            
        doc_vectors.append(mean_vector)

    return doc_vectors

# Generazione delle feature condensate
df_train['vector_pos'] = docs_to_embeddings_pos(df_train['tokens'], df_train['pos'], embeddings_dict)
df_val['vector_pos'] = docs_to_embeddings_pos(df_val['tokens'], df_val['pos'], embeddings_dict)
df_test['vector_pos'] = docs_to_embeddings_pos(df_test['tokens'], df_test['pos'], embeddings_dict)

In [22]:
X_train_pos = np.vstack(df_train['vector_pos'].values)
X_val_pos = np.vstack(df_val['vector_pos'].values)

model_pos = LinearSVC(random_state=42, max_iter=10000)
model_pos.fit(X_train_pos, y_train)
pred_val_pos = model_pos.predict(X_val_pos)

print("--- REPORT PERFORMANCE: EMBEDDING FILTRATI PER POS (VALIDATION) ---")
print(classification_report(y_val, pred_val_pos, zero_division=0))

--- REPORT PERFORMANCE: EMBEDDING FILTRATI PER POS (VALIDATION) ---
              precision    recall  f1-score   support

           0       0.87      0.92      0.89       500
           1       0.92      0.86      0.89       500

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000



### Strategia 4: TF-IDF Weighted Average Pooling
Anziché considerare ogni termine con lo stesso peso specifico, applichiamo una media pesata in cui il contributo di ciascun vettore di parola è moltiplicato per il rispettivo valore **TF-IDF**. Termini rari e informativi avranno un impatto maggiore sul vettore finale del documento rispetto a parole frequenti e cross-documento.

In [23]:
# 1. Inizializzazione e fit del vettorizzatore TF-IDF
tfidf_vect = TfidfVectorizer()
X_train_tfidf = tfidf_vect.fit_transform(df_train['tokens_processed'])
X_val_tfidf = tfidf_vect.transform(df_val['tokens_processed'])
X_test_tfidf = tfidf_vect.transform(df_test['tokens_processed'])

feature_names = tfidf_vect.get_feature_names_out()

# 2. Algoritmo di calcolo per la media pesata
def docs_to_tfidf_embeddings(docs_lists, tfidf_matrix, feature_names, embeddings, emb_dim=128):
    doc_to_array = []
    
    for idx, token_list in enumerate(docs_lists):
        row = tfidf_matrix[idx]
        # Mappatura rapida {parola: score_tfidf} per la riga corrente
        tfidf_dict = {feature_names[col_idx]: val for col_idx, val in zip(row.indices, row.data)}
        
        weighted_vector_sum = np.zeros(emb_dim)
        weight_sum = 0.0
        
        for token in token_list:
            if token in embeddings and token in tfidf_dict:
                weight = tfidf_dict[token]
                weighted_vector_sum += embeddings[token] * weight
                weight_sum += weight
                
        if weight_sum > 0:
            mean_vector = weighted_vector_sum / weight_sum
        else:
            mean_vector = np.zeros(emb_dim)
            
        doc_to_array.append(mean_vector)
        
    return np.vstack(doc_to_array)

print("Estrazione in corso degli embedding pesati con TF-IDF...")
X_train_weighted = docs_to_tfidf_embeddings(df_train['tokens'], X_train_tfidf, feature_names, embeddings_dict)
X_val_weighted = docs_to_tfidf_embeddings(df_val['tokens'], X_val_tfidf, feature_names, embeddings_dict)
X_test_weighted = docs_to_tfidf_embeddings(df_test['tokens'], X_test_tfidf, feature_names, embeddings_dict)

# 3. Classificazione ed esame delle metriche di validazione
model_weighted = LinearSVC(random_state=42, max_iter=10000)
model_weighted.fit(X_train_weighted, y_train)
pred_val_weighted = model_weighted.predict(X_val_weighted)

print("\n--- REPORT PERFORMANCE: EMBEDDING PESATI TF-IDF (VALIDATION) ---")
print(classification_report(y_val, pred_val_weighted, zero_division=0))

Estrazione in corso degli embedding pesati con TF-IDF...

--- REPORT PERFORMANCE: EMBEDDING PESATI TF-IDF (VALIDATION) ---
              precision    recall  f1-score   support

           0       0.88      0.90      0.89       500
           1       0.90      0.88      0.89       500

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000



## 3. Valutazione Finale sul Test Set Blind

Dall'analisi sul Validation Set emerge che i due approcci di Pooling Semplice (**Tokens Embeddings** e **Lemmas Embeddings**) dominano ex-aequo la selezione con una Macro F1 di **0.91** (rispetto allo 0.89 delle varianti POS e TF-IDF). 

Procediamo alla verifica di generalizzazione finale sul Test Set per determinare l'algoritmo vincitore del Task 3. Per garantire la massima correttezza metodologica, introduciamo una pipeline contenente lo **StandardScaler**, tecnica cruciale per stabilizzare i confini di separazione delle SVM operanti su vettori densi non normalizzati.

In [24]:
X_test_tokens = np.vstack(df_test['tokens_vector'].values)
y_test = df_test['label'].values

# Definizione della pipeline ottimizzata con riscaldatore dei coefficienti densi
final_pipeline_tokens = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(random_state=42, max_iter=10000))
])

print("Valutazione finale dell'approccio basato su TOKENS...")
final_pipeline_tokens.fit(X_train_tokens, y_train)
final_pred_tokens = final_pipeline_tokens.predict(X_test_tokens)

print("\n--- REPORT FINALE TOKENS EMBEDDINGS (TEST SET) ---")
print(classification_report(y_test, final_pred_tokens, zero_division=0))

Valutazione finale dell'approccio basato su TOKENS...

--- REPORT FINALE TOKENS EMBEDDINGS (TEST SET) ---
              precision    recall  f1-score   support

           0       0.70      0.93      0.80       500
           1       0.90      0.60      0.72       500

    accuracy                           0.77      1000
   macro avg       0.80      0.77      0.76      1000
weighted avg       0.80      0.77      0.76      1000



In [25]:
X_test_lemmas = np.vstack(df_test['lemmas_vector'].values)

final_pipeline_lemmas = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearSVC(random_state=42, max_iter=10000))
])

print("Valutazione finale dell'approccio basato su LEMMI...")
final_pipeline_lemmas.fit(X_train_lemmas, y_train)
final_pred_lemmas = final_pipeline_lemmas.predict(X_test_lemmas)

print("\n--- REPORT FINALE LEMMAS EMBEDDINGS (TEST SET) ---")
print(classification_report(y_test, final_pred_lemmas, zero_division=0))

Valutazione finale dell'approccio basato su LEMMI...

--- REPORT FINALE LEMMAS EMBEDDINGS (TEST SET) ---
              precision    recall  f1-score   support

           0       0.71      0.94      0.81       500
           1       0.91      0.62      0.74       500

    accuracy                           0.78      1000
   macro avg       0.81      0.78      0.77      1000
weighted avg       0.81      0.78      0.77      1000



### 📊 Analisi Comparativa e Conclusioni finali

